# Simulating Pearl's 2009 SCM Paper
## 'Causal Inference in Statistics: An Overview'
### Pearl, J. (2009). *Statistics Surveys*, 3, 96-146.
### https://ftp.cs.ucla.edu/pub/stat_ser/r350-reprint.pdf

---

This notebook simulates the key concepts and examples from the paper:

1. **Structural Causal Models (SCMs)** - formal definitions and structural equations  
2. **DAGs and d-separation** - reading conditional independence from graphs  
3. **Simpson's Paradox** - how association reverses upon conditioning  
4. **The do-operator** - interventional vs. observational distributions  
5. **Backdoor criterion and adjustment** - controlling for confounding  
6. **Front-door criterion and adjustment** - identification despite hidden confounders  
7. **Mediation analysis** - direct vs. indirect causal effects  


## 0. Setup


In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

from scm import SCM, DiscreteDistribution, d_separated

np.random.seed(42)

# ── Helper: draw a DAG with optional highlighted edges ──────────────────────
def draw_dag(dag, title='', highlight_edges=None, figsize=(6, 3.5), ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        standalone = True
    else:
        standalone = False
    pos = nx.spring_layout(dag, seed=0)
    try:
        from networkx.drawing.nx_agraph import graphviz_layout
        pos = graphviz_layout(dag, prog='dot', args='-Grankdir=LR')
    except Exception:
        pass
    normal_edges = [e for e in dag.edges if not (highlight_edges and e in highlight_edges)]
    hi_edges = highlight_edges or []
    nx.draw_networkx_nodes(dag, pos, node_color='#4C72B0', node_size=900, ax=ax)
    nx.draw_networkx_labels(dag, pos, font_color='white', font_size=12, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(dag, pos, edgelist=normal_edges,
                           edge_color='#555', arrows=True, arrowsize=20,
                           connectionstyle='arc3,rad=0.05', ax=ax)
    if hi_edges:
        nx.draw_networkx_edges(dag, pos, edgelist=hi_edges,
                               edge_color='#DD4444', width=2.5, arrows=True,
                               arrowsize=25, connectionstyle='arc3,rad=0.05', ax=ax)
    ax.set_title(title, fontsize=13)
    ax.axis('off')
    if standalone:
        plt.tight_layout()
        plt.show()

print('Setup complete.')


---
## 1. Structural Causal Models (SCMs) - Core Definitions

**Definition (Pearl 2009, Section 1).**  
A *Structural Causal Model* (SCM) is a 4-tuple  
$$M = \langle U,\, V,\, F,\, P(U) \rangle$$  
where  
* $U$ - exogenous (background) variables with joint distribution $P(U)$,  
* $V$ - endogenous variables,  
* $F = \{f_i\}$ - structural equations $V_i := f_i(\mathrm{Pa}(V_i), U_i)$,  
* $P(U)$ - a probability function over the exogenous variables.  

The structural equations induce a DAG $G$ where a directed edge $X \to Y$
means $X$ is a direct cause of $Y$.

### 1.1 A minimal three-variable SCM

We construct the classic SCM used throughout the paper:

```
Z ---> X ---> Y
```

with structural equations:

$$Z := U_Z, \qquad X := f(Z, U_X), \qquad Y := g(X, U_Y)$$

where $U_Z, U_X, U_Y$ are independent Bernoulli noise variables.


In [ ]:
# ── Structural equations (noise integrated out analytically) ──────────────
# Variables are binary: 0 / 1
# P(Z=1) = 0.5
# P(X=1 | Z) = 0.7 if Z=1 else 0.3
# P(Y=1 | X) = 0.8 if X=1 else 0.2

p_Z   = np.array([0.5, 0.5])                        # P(Z)
p_X_Z = np.array([[0.7, 0.3], [0.3, 0.7]])          # P(X | Z):  [Z, X]
p_Y_X = np.array([[0.8, 0.2], [0.2, 0.8]])          # P(Y | X):  [X, Y]

# Build the joint P(Z, X, Y) by factorisation
joint_chain = np.zeros((2, 2, 2))
for z in range(2):
    for x in range(2):
        for y in range(2):
            joint_chain[z, x, y] = p_Z[z] * p_X_Z[z, x] * p_Y_X[x, y]

assert np.isclose(joint_chain.sum(), 1.0)

dist_chain = DiscreteDistribution(
    variables=['Z', 'X', 'Y'],
    cardinalities={'Z': 2, 'X': 2, 'Y': 2},
    probabilities=joint_chain,
)

dag_chain = nx.DiGraph([('Z', 'X'), ('X', 'Y')])

model_chain = SCM(
    name='Chain Z->X->Y',
    dag=dag_chain,
    joint_dist=dist_chain,
    description='Simple three-node chain. Z is a pre-treatment variable.',
)

draw_dag(dag_chain, title='Chain model: Z -> X -> Y')
print(model_chain)


In [ ]:
print('Marginal P(Z):')
print(dist_chain.marginal(['Z']).to_dataframe().to_string(index=False))
print('\nMarginal P(X):')
print(dist_chain.marginal(['X']).to_dataframe().to_string(index=False))
print('\nMarginal P(Y):')
print(dist_chain.marginal(['Y']).to_dataframe().to_string(index=False))


---
## 2. d-Separation - Reading Conditional Independence from the DAG

**Definition (Pearl 2009, Section 2.3).**  
A set $Z$ *d-separates* $X$ from $Y$ in DAG $G$ if and only if $Z$ blocks
every path between $X$ and $Y$. A path is *blocked* by $Z$ if it contains
either

* a **chain** $A \to B \to C$ or **fork** $A \leftarrow B \to C$ with $B \in Z$, or  
* a **collider** $A \to B \leftarrow C$ with $B \notin Z$ and no descendant of $B$ in $Z$.

d-separation implies *conditional independence* under the Markov condition.

We use the *moral-graph* algorithm:
1. Restrict the DAG to the ancestral subgraph of $X \cup Y \cup Z$.
2. Marry co-parents (connect parents sharing a child).
3. Remove $Z$ nodes.
4. $X \perp Y \mid Z$ iff $X$ and $Y$ are disconnected.


In [ ]:
# ── Fork structure: Z -> X, Z -> Y ────────────────────────────────────────
dag_fork = nx.DiGraph([('Z', 'X'), ('Z', 'Y')])

print('Fork structure: Z -> X, Z -> Y')
print(f'  X _|_ Y (unconditioned)?         {d_separated(dag_fork, ["X"], ["Y"])}')
print(f'  X _|_ Y | Z (conditioned on Z)?  {d_separated(dag_fork, ["X"], ["Y"], ["Z"])}')
print()

# ── Chain structure: X -> Z -> Y ──────────────────────────────────────────
dag_chain2 = nx.DiGraph([('X', 'Z'), ('Z', 'Y')])
print('Chain structure: X -> Z -> Y')
print(f'  X _|_ Y (unconditioned)?         {d_separated(dag_chain2, ["X"], ["Y"])}')
print(f'  X _|_ Y | Z (conditioned on Z)?  {d_separated(dag_chain2, ["X"], ["Y"], ["Z"])}')
print()

# ── Collider structure: X -> Z <- Y ───────────────────────────────────────
dag_collider = nx.DiGraph([('X', 'Z'), ('Y', 'Z')])
print('Collider structure: X -> Z <- Y')
print(f'  X _|_ Y (unconditioned)?         {d_separated(dag_collider, ["X"], ["Y"])}')
print(f'  X _|_ Y | Z (conditioned on Z)?  {d_separated(dag_collider, ["X"], ["Y"], ["Z"])}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

draw_dag(dag_fork,     title='Fork: Z -> X, Z -> Y',       ax=axes[0])
draw_dag(dag_chain2,   title='Chain: X -> Z -> Y',          ax=axes[1])
draw_dag(dag_collider, title='Collider: X -> Z <- Y',       ax=axes[2])

fig.suptitle('Three canonical path structures and d-separation', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('figures/path_structures.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to figures/path_structures.png')


### Interpretation

| Structure | $X \perp Y$ | $X \perp Y \mid Z$ |
|-----------|:-----------:|:-------------------:|
| Fork $Z \to \{X, Y\}$ | No - correlated via $Z$ | Yes - blocked by $Z$ |
| Chain $X \to Z \to Y$ | No - correlated via $Z$ | Yes - blocked by $Z$ |
| Collider $X \to Z \leftarrow Y$ | Yes - independent | No - opened by $Z$ |

The collider row is the famous **explaining away** effect - conditioning on
a common effect induces a spurious dependence between its causes.


---
## 3. Simpson's Paradox

**Example (Pearl 2009, Sections 1 and 3).** A study examines the effect of a drug
on recovery. The raw data show that the drug *hurts* overall, yet *helps*
in every subgroup defined by gender. This paradox arises because **gender
is a common cause** of both drug-taking and recovery - a classic confounder.

```
Gender (G)
  |           \
  v             v
Drug (X) ----> Recovery (Y)
```


In [ ]:
# ── Reproduce Simpson's Paradox numerically ──────────────────────────────
# Classic example: drug appears HARMFUL overall, but BENEFICIAL in each group.
# G=0: healthy people (rarely take drug, good baseline recovery)
# G=1: sick people   (often take drug, poor baseline recovery)
#
# Counts: (G, X): [count_Y0, count_Y1]  (Y=0: no recover, Y=1: recover)
data = {
    (0, 0): [90,  360],  # healthy, no drug:  450 total, 360 recover (80%)
    (0, 1): [5,   45],   # healthy, drug:      50 total,  45 recover (90%)
    (1, 0): [40,  10],   # sick, no drug:      50 total,  10 recover (20%)
    (1, 1): [225, 225],  # sick, drug:        450 total, 225 recover (50%)
}

total = sum(v[0] + v[1] for v in data.values())
print(f'Total population: {total}')

joint_simp = np.zeros((2, 2, 2))  # [G, X, Y]
for (g, x), counts in data.items():
    for y, cnt in enumerate(counts):
        joint_simp[g, x, y] = cnt / total

dist_simp = DiscreteDistribution(
    variables=['G', 'X', 'Y'],
    cardinalities={'G': 2, 'X': 2, 'Y': 2},
    probabilities=joint_simp,
)

# ── Crude association ──────────────────────────────────────────────────────
p_rec_drug   = dist_simp.conditional_p({'Y': 1}, {'X': 1})
p_rec_nodrug = dist_simp.conditional_p({'Y': 1}, {'X': 0})
print(f'\n-- Crude (unadjusted) association --')
print(f'P(recovery | drug)    = {p_rec_drug:.3f}')
print(f'P(recovery | no drug) = {p_rec_nodrug:.3f}')
print(f'Crude risk difference = {p_rec_drug - p_rec_nodrug:+.3f}  <- drug appears HARMFUL')

# ── Within-subgroup associations ───────────────────────────────────────────
print(f'\n-- Stratified by health status --')
for g, label in [(0, 'Healthy'), (1, 'Sick')]:
    p1 = dist_simp.conditional_p({'Y': 1}, {'X': 1, 'G': g})
    p0 = dist_simp.conditional_p({'Y': 1}, {'X': 0, 'G': g})
    print(f'{label}: P(rec|drug) = {p1:.3f},  P(rec|no drug) = {p0:.3f},  diff = {p1-p0:+.3f}  <- drug is BENEFICIAL')

# ── Backdoor-adjusted causal effect ───────────────────────────────────────
dag_simp = nx.DiGraph([('G', 'X'), ('G', 'Y'), ('X', 'Y')])
model_simp = SCM('Simpson', dag_simp, dist_simp)

p_causal_drug   = model_simp.backdoor_adjustment('X', 'Y', ['G'], 1, 1)
p_causal_nodrug = model_simp.backdoor_adjustment('X', 'Y', ['G'], 0, 1)
print(f'\n-- Backdoor-adjusted causal effect --')
print(f'P(Y=1 | do(X=1)) = {p_causal_drug:.3f}')
print(f'P(Y=1 | do(X=0)) = {p_causal_nodrug:.3f}')
print(f'Causal risk difference = {p_causal_drug - p_causal_nodrug:+.3f}  <- drug is truly BENEFICIAL')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

draw_dag(dag_simp,
         title='DAG: Health status confounds Drug -> Recovery',
         highlight_edges=[('G', 'X'), ('G', 'Y')],
         ax=axes[0])

ax = axes[1]
categories = ['Crude\n(drug)', 'Crude\n(no drug)',
               'Healthy\n(drug)', 'Healthy\n(no drug)',
               'Sick\n(drug)', 'Sick\n(no drug)',
               'Causal\ndo(drug)', 'Causal\ndo(no drug)']
values = [
    p_rec_drug, p_rec_nodrug,
    dist_simp.conditional_p({'Y': 1}, {'X': 1, 'G': 0}),
    dist_simp.conditional_p({'Y': 1}, {'X': 0, 'G': 0}),
    dist_simp.conditional_p({'Y': 1}, {'X': 1, 'G': 1}),
    dist_simp.conditional_p({'Y': 1}, {'X': 0, 'G': 1}),
    p_causal_drug, p_causal_nodrug,
]
colors = ['#DD4444', '#4444DD',
          '#DD4444', '#4444DD',
          '#DD4444', '#4444DD',
          '#44AA44', '#88CC88']
ax.bar(categories, values, color=colors, edgecolor='white', width=0.65)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
ax.set_ylim(0, 1)
ax.set_ylabel('P(Recovery = 1)')
ax.set_title('Recovery rates: crude vs. stratified vs. causal')
patches = [mpatches.Patch(color='#DD4444', label='Drug'),
           mpatches.Patch(color='#4444DD', label='No drug'),
           mpatches.Patch(color='#44AA44', label='do(Drug) - causal')]
ax.legend(handles=patches, loc='lower right')

plt.tight_layout()
plt.savefig('figures/simpsons_paradox.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to figures/simpsons_paradox.png')


### Interpretation

The crude association is **negative** (drug appears harmful), but once we
adjust for the confounder *Health status* - either by stratification or by the
backdoor formula - the causal effect is **positive**.

The confusion arises because sick people are more likely to take the drug
yet have lower baseline recovery rates. The backdoor criterion tells us
which variable(s) to adjust for and the backdoor formula gives the correct
causal estimate.


---
## 4. The do-Operator and Interventional Distributions

**Definition (Pearl 2009, Section 3.1).**  
The *do-operator* $\mathrm{do}(X = x)$ represents a surgical intervention that
sets $X$ to value $x$ while preserving the rest of the causal mechanism.
Formally, it induces the *mutilated model* $M_x$ where the structural
equation for $X$ is replaced by the constant $x$ (all incoming edges to
$X$ are removed).

The key distinction:

$$P(Y = y \mid X = x) \qquad \text{(observational)}$$
$$P(Y = y \mid \mathrm{do}(X = x)) \qquad \text{(interventional)}$$

These coincide only when $X$ has no confounders.


In [ ]:
# ── Confounded model: U -> X, U -> Y, X -> Y (U is hidden) ────────────────
p_Y_X1_U0, p_Y_X0_U0 = 0.7, 0.3    # U=0
p_Y_X1_U1, p_Y_X0_U1 = 0.9, 0.5    # U=1

p_U = np.array([0.5, 0.5])
p_X_U = np.array([[0.9, 0.1],   # U=0
                   [0.1, 0.9]])  # U=1

p_Y_XU = np.array([[[1-p_Y_X0_U0, p_Y_X0_U0],
                     [1-p_Y_X1_U0, p_Y_X1_U0]],
                    [[1-p_Y_X0_U1, p_Y_X0_U1],
                     [1-p_Y_X1_U1, p_Y_X1_U1]]])

joint_conf = np.zeros((2, 2, 2))  # [U, X, Y]
for u in range(2):
    for x in range(2):
        for y in range(2):
            joint_conf[u, x, y] = p_U[u] * p_X_U[u, x] * p_Y_XU[u, x, y]

dist_conf_full = DiscreteDistribution(
    ['U', 'X', 'Y'], {'U': 2, 'X': 2, 'Y': 2}, joint_conf
)
dist_obs = dist_conf_full.marginal(['X', 'Y'])

obs_drug   = dist_obs.conditional_p({'Y': 1}, {'X': 1})
obs_nodrug = dist_obs.conditional_p({'Y': 1}, {'X': 0})

# True causal effect: P(Y=1|do(X=x)) = sum_u P(Y=1|X=x,U=u)*P(U=u)
true_do1 = sum(p_U[u] * p_Y_XU[u, 1, 1] for u in range(2))
true_do0 = sum(p_U[u] * p_Y_XU[u, 0, 1] for u in range(2))

print('-- Confounded model: U -> X, U -> Y, X -> Y (U hidden) --')
print(f'Observed  P(Y=1|X=1)       = {obs_drug:.3f}')
print(f'Observed  P(Y=1|X=0)       = {obs_nodrug:.3f}')
print(f'Obs risk diff              = {obs_drug - obs_nodrug:+.3f}  (BIASED)')
print()
print(f'True P(Y=1|do(X=1))        = {true_do1:.3f}')
print(f'True P(Y=1|do(X=0))        = {true_do0:.3f}')
print(f'True causal diff           = {true_do1 - true_do0:+.3f}  (UNBIASED)')


---
## 5. Backdoor Criterion and Adjustment

**Theorem (Pearl 2009, Theorem 3.3.2 - Backdoor Adjustment).**  
If a set $Z$ satisfies the backdoor criterion relative to $(X, Y)$, then:

$$P(Y = y \mid \mathrm{do}(X = x)) = \sum_z P(Y = y \mid X = x,\, Z = z)\, P(Z = z)$$

**Backdoor criterion (Pearl 2009, Definition 3.3.1).**  
A set $Z$ satisfies the backdoor criterion relative to $(X, Y)$ in DAG $G$ iff:
1. No node in $Z$ is a descendant of $X$.
2. $Z$ blocks every backdoor path from $X$ to $Y$.

### 5.1 Drug-recovery example with observed confounder


In [ ]:
# ── Model: Z (age) confounds X (drug) and Y (recovery); X -> Y ──────────
params = {
    'p_Z1': 0.40,
    'p_X_Z': {0: 0.25, 1: 0.75},
    'p_Y_XZ': {(0,0): 0.40, (1,0): 0.60,
               (0,1): 0.20, (1,1): 0.50},
}

joint_bd = np.zeros((2, 2, 2))  # [Z, X, Y]
for z in range(2):
    pz = params['p_Z1'] if z == 1 else 1 - params['p_Z1']
    for x in range(2):
        px_z = params['p_X_Z'][z] if x == 1 else 1 - params['p_X_Z'][z]
        for y in range(2):
            py_xz = params['p_Y_XZ'][(x, z)] if y == 1 else 1 - params['p_Y_XZ'][(x, z)]
            joint_bd[z, x, y] = pz * px_z * py_xz

dist_bd = DiscreteDistribution(['Z', 'X', 'Y'], {'Z': 2, 'X': 2, 'Y': 2}, joint_bd)
dag_bd  = nx.DiGraph([('Z', 'X'), ('Z', 'Y'), ('X', 'Y')])
model_bd = SCM('Backdoor example', dag_bd, dist_bd)

print('Z satisfies backdoor for (X, Y):', model_bd.satisfies_backdoor('X', 'Y', ['Z']))

crude_1 = dist_bd.conditional_p({'Y': 1}, {'X': 1})
crude_0 = dist_bd.conditional_p({'Y': 1}, {'X': 0})
causal_1 = model_bd.backdoor_adjustment('X', 'Y', ['Z'], 1, 1)
causal_0 = model_bd.backdoor_adjustment('X', 'Y', ['Z'], 0, 1)

# Ground truth
pz1 = params['p_Z1']
true_do1_bd = params['p_Y_XZ'][(1,0)] * (1 - pz1) + params['p_Y_XZ'][(1,1)] * pz1
true_do0_bd = params['p_Y_XZ'][(0,0)] * (1 - pz1) + params['p_Y_XZ'][(0,1)] * pz1

print(f'\nP(Y=1|X=1)          = {crude_1:.4f}   (observational - biased)')
print(f'P(Y=1|X=0)          = {crude_0:.4f}   (observational - biased)')
print(f'Crude difference    = {crude_1-crude_0:+.4f}')
print()
print(f'P(Y=1|do(X=1))      = {causal_1:.4f}   (backdoor adjustment)')
print(f'P(Y=1|do(X=0))      = {causal_0:.4f}   (backdoor adjustment)')
print(f'Causal difference   = {causal_1-causal_0:+.4f}')
print()
print(f'True P(Y=1|do(X=1)) = {true_do1_bd:.4f}   (model parameters)')
print(f'True P(Y=1|do(X=0)) = {true_do0_bd:.4f}   (model parameters)')
print(f'Backdoor = True:      {np.isclose(causal_1, true_do1_bd) and np.isclose(causal_0, true_do0_bd)}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

draw_dag(dag_bd,
         title='Backdoor: Z confounds X -> Y',
         highlight_edges=[('Z', 'X'), ('Z', 'Y')],
         ax=axes[0])

ax = axes[1]
x_pos = np.arange(2)
w = 0.3
ax.bar(x_pos - w/2, [crude_0, crude_1],   width=w, label='Observational P(Y=1|X)',
       color='#DD8844', edgecolor='white')
ax.bar(x_pos + w/2, [causal_0, causal_1], width=w, label='Causal P(Y=1|do(X))',
       color='#44AA44', edgecolor='white')
ax.set_xticks(x_pos)
ax.set_xticklabels(['No Drug (X=0)', 'Drug (X=1)'])
ax.set_ylabel('P(Recovery = 1)')
ax.set_ylim(0, 0.8)
ax.set_title('Observational vs. causal effect of drug on recovery')
ax.legend()

plt.tight_layout()
plt.savefig('figures/backdoor_adjustment.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to figures/backdoor_adjustment.png')


---
## 6. Front-Door Criterion and Adjustment

**Theorem (Pearl 2009, Theorem 3.3.4 - Front-Door Adjustment).**  
If a set $M$ satisfies the front-door criterion relative to $(X, Y)$ and
$P(X, M) > 0$, then the causal effect is identifiable:

$$P(Y = y \mid \mathrm{do}(X = x)) =
   \sum_m P(M = m \mid X = x)
   \sum_{x'} P(Y = y \mid M = m,\, X = x')\, P(X = x')$$

The front-door criterion applies when there is an **unobserved confounder**
between $X$ and $Y$, but a mediator $M$ is observed such that:
1. $M$ intercepts all directed paths $X \to Y$.
2. No unblocked backdoor path exists from $X$ to $M$.
3. All backdoor paths from $M$ to $Y$ are blocked by $X$.

### 6.1 Smoking -> Tar -> Cancer with hidden confounder

```
    U (hidden)
   /            \
  v               v
Smoking (X) ---> Tar (M) ---> Cancer (Y)
```

$U$ is a genetic factor predisposing to both smoking and cancer.


In [ ]:
# ── Define the full model with U (hidden) ────────────────────────────────
p_U_fd   = np.array([0.5, 0.5])
p_X_U_fd = {0: 0.15, 1: 0.70}        # P(X=1 | U)
p_M_X_fd = {0: 0.05, 1: 0.90}        # P(M=1 | X)
p_Y_MU_fd = {(0,0): 0.02, (1,0): 0.20,
              (0,1): 0.20, (1,1): 0.65}  # P(Y=1 | M, U)

joint_fd_full = np.zeros((2, 2, 2, 2))  # [U, X, M, Y]
for u in range(2):
    for x in range(2):
        for m in range(2):
            for y in range(2):
                px_u  = p_X_U_fd[u] if x == 1 else 1 - p_X_U_fd[u]
                pm_x  = p_M_X_fd[x] if m == 1 else 1 - p_M_X_fd[x]
                py_mu = p_Y_MU_fd[(m, u)] if y == 1 else 1 - p_Y_MU_fd[(m, u)]
                joint_fd_full[u, x, m, y] = p_U_fd[u] * px_u * pm_x * py_mu

dist_fd_full = DiscreteDistribution(
    ['U', 'X', 'M', 'Y'], {'U': 2, 'X': 2, 'M': 2, 'Y': 2}, joint_fd_full
)
dist_fd_obs = dist_fd_full.marginal(['X', 'M', 'Y'])

dag_fd_obs = nx.DiGraph([('X', 'M'), ('M', 'Y')])
model_fd = SCM('Front-door (Smoking/Tar/Cancer)', dag_fd_obs, dist_fd_obs)

print('Front-door criterion satisfied for X->Y via M:',
      model_fd.satisfies_frontdoor('X', 'Y', ['M']))

dist_xy = dist_fd_obs.marginal(['X', 'Y'])
obs_smoke   = dist_xy.conditional_p({'Y': 1}, {'X': 1})
obs_nosmoke = dist_xy.conditional_p({'Y': 1}, {'X': 0})

def true_do_fd(x_val):
    result = 0.0
    for u in range(2):
        for m in range(2):
            pm_x  = p_M_X_fd[x_val] if m == 1 else 1 - p_M_X_fd[x_val]
            py_mu = p_Y_MU_fd[(m, u)]
            result += p_U_fd[u] * pm_x * py_mu
    return result

true_fd1 = true_do_fd(1)
true_fd0 = true_do_fd(0)

fd_1 = model_fd.frontdoor_adjustment('X', 'Y', ['M'], 1, 1)
fd_0 = model_fd.frontdoor_adjustment('X', 'Y', ['M'], 0, 1)

print(f'\nP(Y=1|X=1)          = {obs_smoke:.4f}   (observational - confounded)')
print(f'P(Y=1|X=0)          = {obs_nosmoke:.4f}   (observational - confounded)')
print(f'Obs risk diff       = {obs_smoke-obs_nosmoke:+.4f}')
print()
print(f'P(Y=1|do(X=1))      = {fd_1:.4f}   (front-door adjustment)')
print(f'P(Y=1|do(X=0))      = {fd_0:.4f}   (front-door adjustment)')
print(f'Causal diff (FD)    = {fd_1-fd_0:+.4f}')
print()
print(f'True P(Y=1|do(X=1)) = {true_fd1:.4f}   (ground truth)')
print(f'True P(Y=1|do(X=0)) = {true_fd0:.4f}   (ground truth)')
print(f'Front-door = True:   {np.isclose(fd_1, true_fd1) and np.isclose(fd_0, true_fd0)}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: DAG with hidden confounder shown conceptually
ax = axes[0]
pos_fd = {'X': (0, 0), 'M': (1, 0), 'Y': (2, 0)}
dag_vis = dag_fd_obs.copy()
nx.draw_networkx_nodes(dag_vis, pos_fd, node_color='#4C72B0', node_size=900, ax=ax)
nx.draw_networkx_labels(dag_vis, pos_fd,
                        font_color='white', font_size=12, font_weight='bold', ax=ax)
nx.draw_networkx_edges(dag_vis, pos_fd, edgelist=[('X','M'),('M','Y')],
                       edge_color='#555', arrows=True, arrowsize=20,
                       connectionstyle='arc3,rad=0.0', ax=ax)
# Draw hidden confounder U as a dashed curved path X <-> Y
ax.annotate('', xy=(2, 0.15), xytext=(0, 0.15),
            arrowprops=dict(arrowstyle='<->', color='#AA4444',
                            linestyle='dashed', lw=1.8,
                            connectionstyle='arc3,rad=-0.4'))
ax.text(1, 0.55, 'U (hidden)', ha='center', color='#AA4444', fontsize=10)
ax.set_xlim(-0.4, 2.4)
ax.set_ylim(-0.4, 0.9)
ax.set_title('Front-door model\nX->M->Y, hidden confounder U (dashed)', fontsize=12)
ax.axis('off')

# Right: bar chart
ax2 = axes[1]
x_pos = np.arange(2)
w = 0.3
ax2.bar(x_pos - w/2, [obs_nosmoke, obs_smoke], width=w,
        label='Observational P(Y=1|X)', color='#DD8844', edgecolor='white')
ax2.bar(x_pos + w/2, [fd_0, fd_1],            width=w,
        label='Causal P(Y=1|do(X)) - front-door', color='#4477AA', edgecolor='white')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(['No Smoking (X=0)', 'Smoking (X=1)'])
ax2.set_ylabel('P(Cancer = 1)')
ax2.set_ylim(0, 0.55)
ax2.set_title('Observational vs. causal effect\n(front-door, U hidden)')
ax2.legend()

plt.tight_layout()
plt.savefig('figures/frontdoor_adjustment.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to figures/frontdoor_adjustment.png')


---
## 7. Mediation Analysis - Direct vs. Indirect Effects

**Section (Pearl 2009, Section 4).** Mediation analysis decomposes the total causal
effect of $X$ on $Y$ into:

* **Natural Direct Effect (NDE):** Effect of $X$ on $Y$ not through mediator $M$
* **Natural Indirect Effect (NIE):** Effect of $X$ on $Y$ mediated through $M$
* **Total Effect:** $\mathrm{TE} = \mathrm{NDE} + \mathrm{NIE}$

For a **linear** Gaussian SCM:
$$X = U_X,\quad M = \alpha X + U_M,\quad Y = \beta M + \gamma X + U_Y$$
the path coefficients are: $\mathrm{NDE} = \gamma$, $\mathrm{NIE} = \alpha\beta$.


In [ ]:
np.random.seed(42)

alpha = 0.7   # X -> M
beta  = 0.5   # M -> Y
gamma = 0.3   # X -> Y (direct)

n = 50_000
U_X = np.random.normal(0, 1, n)
U_M = np.random.normal(0, 1, n)
U_Y = np.random.normal(0, 1, n)

X_sim = U_X
M_sim = alpha * X_sim + U_M
Y_sim = beta  * M_sim + gamma * X_sim + U_Y

# OLS regression
alpha_hat = np.cov(X_sim, M_sim)[0, 1] / np.var(X_sim)
A = np.column_stack([X_sim, M_sim, np.ones(n)])
coefs, _, _, _ = np.linalg.lstsq(A, Y_sim, rcond=None)
gamma_hat, beta_hat = coefs[0], coefs[1]

NDE_true = gamma
NIE_true = alpha * beta
TE_true  = NDE_true + NIE_true

NDE_est = gamma_hat
NIE_est = alpha_hat * beta_hat
TE_est  = NDE_est + NIE_est

print('Linear mediation (Gaussian SCM)')
print(f'Parameters: alpha={alpha}, beta={beta}, gamma={gamma}')
print()
print(f'{"Effect":<25} {"True":>8} {"Estimated":>12}')
print('-' * 47)
print(f'{"NDE (direct)  gamma":<25} {NDE_true:>8.4f} {NDE_est:>12.4f}')
print(f'{"NIE (indirect) alpha*beta":<25} {NIE_true:>8.4f} {NIE_est:>12.4f}')
print(f'{"TE  (total) gamma+alpha*beta":<25} {TE_true:>8.4f} {TE_est:>12.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
dag_med = nx.DiGraph([('X', 'M'), ('M', 'Y'), ('X', 'Y')])
pos_med = {'X': (0, 0), 'M': (1, 1), 'Y': (2, 0)}
nx.draw_networkx_nodes(dag_med, pos_med, node_color='#4C72B0', node_size=900, ax=ax)
nx.draw_networkx_labels(dag_med, pos_med, font_color='white',
                        font_size=12, font_weight='bold', ax=ax)
for (u, v), label, rad, color in [
    (('X', 'M'), f'alpha={alpha}',  0.15, '#4477AA'),
    (('M', 'Y'), f'beta={beta}',    0.15, '#4477AA'),
    (('X', 'Y'), f'gamma={gamma}', -0.25, '#DD4444'),
]:
    x0, y0 = pos_med[u]
    x1, y1 = pos_med[v]
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2,
                                connectionstyle=f'arc3,rad={rad}'))
    mx, my = (x0 + x1) / 2, (y0 + y1) / 2
    ax.text(mx, my + 0.05 * rad / abs(rad + 1e-9), label,
            ha='center', va='center', fontsize=10, color=color)
ax.set_xlim(-0.4, 2.4)
ax.set_ylim(-0.5, 1.5)
ax.set_title(f'Mediation DAG\nDirect (red): gamma={gamma}  Indirect (blue): alpha*beta={alpha*beta:.2f}')
ax.axis('off')

ax2 = axes[1]
labels = ['NDE (direct)', 'NIE (indirect)', 'Total Effect']
true_vals = [NDE_true, NIE_true, TE_true]
est_vals  = [NDE_est,  NIE_est,  TE_est]
x_pos = np.arange(len(labels))
w = 0.3
ax2.bar(x_pos - w/2, true_vals, width=w, label='True',      color='#4C72B0', edgecolor='white')
ax2.bar(x_pos + w/2, est_vals,  width=w, label='Estimated', color='#DD8844', edgecolor='white')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(labels)
ax2.set_ylabel('Effect size')
ax2.set_title('Mediation: Direct vs. Indirect vs. Total Effect')
ax2.legend()

plt.tight_layout()
plt.savefig('figures/mediation_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to figures/mediation_analysis.png')


---
## 8. Summary: Pearl's Ladder of Causation

Pearl's *Ladder of Causation* (2009, Section 1) organises causal reasoning into three rungs:

| Rung | Activity | Question | Tool |
|------|----------|----------|------|
| 1 | **Association** | What if I see X? | $P(Y \mid X)$ - conditional probability |
| 2 | **Intervention** | What if I do X? | $P(Y \mid \mathrm{do}(X))$ - do-operator |
| 3 | **Counterfactual** | What if I had done X? | $P(Y_x \mid X')$ - potential outcomes |

This notebook demonstrated rungs 1 and 2 through concrete simulations.


In [ ]:
summary = pd.DataFrame([
    {'Section': "3. Simpson's Paradox",
     'Model': 'G -> X, G -> Y, X -> Y',
     'Crude effect': f'{p_rec_drug - p_rec_nodrug:+.3f}',
     'Causal do-effect': f'{p_causal_drug - p_causal_nodrug:+.3f}',
     'Method': 'Backdoor adj. (Z=Gender)'},
    {'Section': '5. Backdoor adjustment',
     'Model': 'Z -> X, Z -> Y, X -> Y',
     'Crude effect': f'{crude_1 - crude_0:+.4f}',
     'Causal do-effect': f'{causal_1 - causal_0:+.4f}',
     'Method': 'Backdoor adj. (Z=Age)'},
    {'Section': '6. Front-door adjustment',
     'Model': 'X -> M -> Y, U -> X, U -> Y (hidden)',
     'Crude effect': f'{obs_smoke - obs_nosmoke:+.4f}',
     'Causal do-effect': f'{fd_1 - fd_0:+.4f}',
     'Method': 'Front-door adj. (M=Tar)'},
    {'Section': '7. Mediation analysis',
     'Model': 'X -> M -> Y, X -> Y (linear)',
     'Crude effect': f'{TE_true:+.4f}',
     'Causal do-effect': f'NDE={NDE_true:.2f}, NIE={NIE_true:.2f}',
     'Method': 'Path coefficient decomposition'},
])
print(summary.to_string(index=False))


---
## References

Pearl, J. (2009). Causal inference in statistics: An overview.  
*Statistics Surveys*, **3**, 96-146.  
https://ftp.cs.ucla.edu/pub/stat_ser/r350-reprint.pdf

Pearl, J., Glymour, M., and Jewell, N. P. (2016).  
*Causal Inference in Statistics: A Primer*. Wiley.

Pearl, J. (2000). *Causality: Models, Reasoning, and Inference*. Cambridge University Press.
